# Colab bootstrap — NNDL Saliency Project

Esegui queste celle in ordine ogni volta che apri una nuova sessione Colab.
**Regola d'oro:** salva SEMPRE i checkpoint su Drive, non solo sul disco della VM — la VM viene distrutta alla disconnessione.

## 1. Verifica GPU
Runtime > Change runtime type > GPU, poi esegui questa cella.

In [ ]:
!nvidia-smi

## 2. Monta Google Drive (per checkpoint persistenti)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/nndl-saliency/checkpoints'
DATA_DIR = '/content/drive/MyDrive/nndl-saliency/data'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Data dir:', DATA_DIR)

## 3. Clona il repository

In [ ]:
REPO_URL = 'https://github.com/<org-o-utente>/nndl-saliency.git'  # <-- sostituire
!git clone $REPO_URL /content/nndl-saliency
%cd /content/nndl-saliency

## 4. Installa le dipendenze mancanti

In [ ]:
!pip install -q -r requirements-colab.txt

## 5. Credenziali Kaggle (solo la prima volta / se non già su Drive)
Carica `kaggle.json` quando richiesto (Kaggle > Settings > Create New Token).

In [ ]:
import os
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_drive = '/content/drive/MyDrive/nndl-saliency/kaggle.json'
if os.path.exists(kaggle_json_drive):
    !cp "$kaggle_json_drive" ~/.kaggle/kaggle.json
else:
    from google.colab import files
    uploaded = files.upload()  # carica kaggle.json
    !mv kaggle.json ~/.kaggle/kaggle.json
    !cp ~/.kaggle/kaggle.json "$kaggle_json_drive"  # salva su Drive per la prossima volta
!chmod 600 ~/.kaggle/kaggle.json

## 6. Download + audit del dataset (una tantum, poi resta su Drive)

In [ ]:
!python scripts/download_salicon.py --output_dir "$DATA_DIR"
!python scripts/audit_dataset.py --data_dir "$DATA_DIR"

## 6bis. Copia il dataset su disco locale della VM (IMPORTANTE)

Leggere migliaia di JPEG piccoli direttamente da Drive montato e' molto lento: e' il vero collo di bottiglia su Colab free, non il calcolo. **Farlo a inizio di OGNI sessione**, prima di lanciare un training.

In [ ]:
import shutil, os, time

LOCAL_DATA_DIR = '/content/data_local'  # deve combaciare con configs/data.yaml -> colab_local_cache_dir

if not os.path.exists(LOCAL_DATA_DIR):
    t0 = time.time()
    shutil.copytree(DATA_DIR, LOCAL_DATA_DIR)
    print(f'Copiato {DATA_DIR} -> {LOCAL_DATA_DIR} in {time.time()-t0:.1f}s')
else:
    print(f'{LOCAL_DATA_DIR} gia\' presente, salto la copia (VM ancora attiva dalla sessione precedente).')

# Da qui in poi, train.py / evaluate.py devono puntare a LOCAL_DATA_DIR, non a DATA_DIR.

## 7. Da qui in poi: training / evaluation
Verranno aggiunte celle per `scripts/train.py` e `scripts/evaluate.py` a partire dal giorno 3.